# 2.08 值函数近似与 DQN

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/02_rl/2.08_value_approximation_dqn.ipynb)


## 为什么要学

表格方法给每个 $(s,a)$ 存一个数。但机器人的状态是 224×224 的图像加十几维关节角，状态空间是连续的、无限大的。解决办法：**用一个带参数的函数 $\hat q(s,a;\mathbf{w})$ 来近似 q 值**，比如神经网络。

DQN（Deep Q-Network，2015 年在 Atari 游戏上达到人类水平）是第一个成功的深度 RL 算法。它的两个关键技巧，**经验回放**和**目标网络**，在 SAC、TD3、RLT 里原样保留。对应书中第 8 章。

---

## 1. 值函数近似的目标

用 $\hat v(s;\mathbf{w})$ 近似 $v_\pi(s)$，目标是最小化

$$
J(\mathbf{w})=\mathbb{E}\Big[\big(v_\pi(S)-\hat v(S;\mathbf{w})\big)^2\Big]
$$

期望是对状态的分布取的（通常是策略长期运行下的**平稳分布**：经常访问的状态更重要）。梯度下降：

$$
\mathbf{w}\leftarrow\mathbf{w}+\alpha\big(v_\pi(s_t)-\hat v(s_t;\mathbf{w})\big)\nabla_{\mathbf{w}}\hat v(s_t;\mathbf{w})
$$

$v_\pi(s_t)$ 不知道，用 TD 目标代替，得到 **TD-线性/TD-函数近似**算法：

$$
\mathbf{w}\leftarrow\mathbf{w}+\alpha\big(r_{t+1}+\gamma\hat v(s_{t+1};\mathbf{w})-\hat v(s_t;\mathbf{w})\big)\nabla_{\mathbf{w}}\hat v(s_t;\mathbf{w})
$$

注意：目标里也有 $\mathbf{w}$，但我们**不对它求导**（把它当常数），所以这叫"半梯度"（semi-gradient）方法。代码里就是对目标做 `detach()` / `no_grad()`。

### 例子：用多项式特征近似网格世界的值

书中用 $\hat v(s;\mathbf{w})=\boldsymbol\phi(s)^\top\mathbf{w}$，其中 $\boldsymbol\phi(s)$ 是格子坐标 $(x,y)$ 的多项式特征。**表格方法其实是特征为 one-hot 的线性近似**。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from vla_ladder.envs import GridWorld, CartPole
from vla_ladder.utils import moving_average, set_seed

env_g = GridWorld()
gamma_g = 0.9
pi_rand = np.full((25, 5), 0.2)
v_true = np.linalg.solve(np.eye(25) - gamma_g * np.einsum("sa,sat->st", pi_rand, env_g.P), (pi_rand * env_g.R).sum(1))

def poly_features(s, order):
    r, c = env_g.to_rc(s)
    x, y = c / 4 * 2 - 1, r / 4 * 2 - 1           # 归一化到 [-1, 1]
    return np.array([x ** i * y ** j for i in range(order + 1) for j in range(order + 1 - i)])

rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, order in zip(axes, [1, 3, 5]):
    w = np.zeros(len(poly_features(0, order)))
    s = env_g.reset(0)
    for t in range(100_000):
        a = rng.choice(5, p=pi_rand[s]); ns, r = env_g.step(a)
        phi, phi_n = poly_features(s, order), poly_features(ns, order)
        w += 0.005 * (r + gamma_g * phi_n @ w - phi @ w) * phi        # TD-线性
        s = ns
    v_hat = np.array([poly_features(x, order) @ w for x in range(25)])
    env_g.plot(V=v_hat, title=f"poly order {order}: {len(w)} params, err={np.abs(v_hat - v_true).mean():.2f}", ax=ax)
plt.tight_layout(); plt.show()

阶数越高（参数越多），近似越准，但并不能做到完美。**函数近似用少量参数换来了泛化能力，代价是近似误差**。神经网络就是一个超级灵活的"特征 + 线性层"。

---

## 2. DQN：Q-learning + 神经网络

Q-learning 求解贝尔曼最优方程。用神经网络 $\hat q(s,a;\mathbf{w})$ 时，定义损失

$$
J(\mathbf{w})=\mathbb{E}\Big[\Big(\underbrace{R+\gamma\max_{a'}\hat q(S',a';\mathbf{w}_T)}_{\text{TD 目标}}-\hat q(S,A;\mathbf{w})\Big)^2\Big]
$$

### 技巧一：目标网络（target network）$\mathbf{w}_T$

如果目标也用 $\mathbf{w}$，每更新一次参数，目标就跟着变，相当于"追着自己的尾巴跑"，训练不稳定。DQN 维护一份**目标网络**参数 $\mathbf{w}_T$：

- 计算 TD 目标时用 $\mathbf{w}_T$（不求梯度）；
- 每隔 $C$ 步把 $\mathbf{w}$ 复制给 $\mathbf{w}_T$（硬更新），或每步做软更新 $\mathbf{w}_T\leftarrow\tau\mathbf{w}+(1-\tau)\mathbf{w}_T$（SAC/TD3/RLT 用这种，即 Polyak 平均）。

### 技巧二：经验回放（experience replay）

把交互得到的 $(s,a,r,s',\text{done})$ 存进一个**回放缓冲区**，训练时从中**均匀随机**采样小批量。好处：

1. 打破时间相关性：连续的样本高度相关，违反 SGD 的独立同分布假设；
2. 一条数据可以被反复使用：**样本效率高**（这对真机 RL 至关重要）；
3. 可以这样做的前提是 Q-learning 是 **off-policy** 的：旧策略产生的数据也能用。

### 回合结束的处理：terminated vs truncated

- **terminated**（真正结束，比如杆倒了）：没有未来，TD 目标 $=r$；
- **truncated**（因为时间上限被截断，比如 500 步到了）：其实还有未来，TD 目标仍应是 $r+\gamma\max q(s')$。

这个区别很容易写错。真机任务中"超时"属于 truncated，"插入成功/失败"属于 terminated。

---

## 3. 实现 DQN（在 CartPole 上）

In [ ]:
env = CartPole()
obs, _ = env.reset(seed=0)
print("观测:", obs, "（小车位置、速度、杆角度、角速度）  动作数:", env.n_actions)

### 练习 1：回放缓冲区

用固定大小的 numpy 数组实现**环形缓冲区**：满了以后新数据覆盖最旧的数据。

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity, obs_dim):
        self.capacity = capacity
        self.obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.next_obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.terminated = np.zeros(capacity, dtype=np.float32)
        self.ptr, self.size = 0, 0      # ptr: 下一个写入位置；size: 当前有效数据量

    def add(self, o, a, r, no, term):
        # TODO: 写入 self.ptr 位置；ptr 前进一格并对 capacity 取模；size 不超过 capacity
        raise NotImplementedError

    def sample(self, batch_size, rng):
        # TODO: 从 [0, size) 均匀采样 batch_size 个下标，返回 dict（值转成 torch 张量）
        raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
class ReplayBuffer:
    def __init__(self, capacity, obs_dim):
        self.capacity = capacity
        self.obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.next_obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.terminated = np.zeros(capacity, dtype=np.float32)
        self.ptr, self.size = 0, 0

    def add(self, o, a, r, no, term):
        i = self.ptr
        self.obs[i], self.actions[i], self.rewards[i], self.next_obs[i], self.terminated[i] = o, a, r, no, term
        self.ptr = (self.ptr + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size, rng):
        idx = rng.integers(0, self.size, size=batch_size)
        return {
            "obs": torch.as_tensor(self.obs[idx]),
            "actions": torch.as_tensor(self.actions[idx]),
            "rewards": torch.as_tensor(self.rewards[idx]),
            "next_obs": torch.as_tensor(self.next_obs[idx]),
            "terminated": torch.as_tensor(self.terminated[idx]),
        }

In [ ]:
# ✅ 自测
buf = ReplayBuffer(5, 4)
for k in range(7):
    buf.add(np.full(4, k), k % 2, float(k), np.full(4, k + 1), 0.0)
assert buf.size == 5 and buf.ptr == 2
assert set(buf.rewards.tolist()) == {2.0, 3.0, 4.0, 5.0, 6.0}   # 最早的 0、1 被覆盖了
b = buf.sample(3, np.random.default_rng(0))
assert b["obs"].shape == (3, 4) and b["actions"].dtype == torch.int64
print("通过 ✅")

### 练习 2：DQN 损失

In [ ]:
def dqn_loss(q_net, target_net, batch, gamma):
    """返回标量损失。batch 字段见 ReplayBuffer.sample"""
    # TODO:
    # 1) q_sa = q_net(obs) 中取出所执行动作的 q 值，形状 (B,)
    # 2) 在 no_grad 下：next_max = target_net(next_obs).max(1).values
    #    target = rewards + gamma * (1 - terminated) * next_max
    # 3) 返回 MSE（或 Huber：F.smooth_l1_loss）
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def dqn_loss(q_net, target_net, batch, gamma):
    """返回标量损失。batch 字段见 ReplayBuffer.sample"""
    q_all = q_net(batch["obs"])                                           # (B, A)
    q_sa = q_all.gather(1, batch["actions"][:, None]).squeeze(1)          # (B,)
    with torch.no_grad():
        next_max = target_net(batch["next_obs"]).max(1).values            # (B,)
        target = batch["rewards"] + gamma * (1 - batch["terminated"]) * next_max
    return F.mse_loss(q_sa, target)

In [ ]:
# ✅ 自测：terminated=1 时目标只等于奖励
qn = nn.Linear(4, 2); tn = nn.Linear(4, 2)
batch = {"obs": torch.randn(4, 4), "actions": torch.tensor([0, 1, 0, 1]), "rewards": torch.ones(4),
         "next_obs": torch.randn(4, 4), "terminated": torch.ones(4)}
with torch.no_grad():
    q_sa = qn(batch["obs"]).gather(1, batch["actions"][:, None]).squeeze(1)
assert torch.isclose(dqn_loss(qn, tn, batch, 0.99), ((q_sa - 1) ** 2).mean())
print("通过 ✅")

### 训练 DQN

In [ ]:
def train_dqn(total_steps=30_000, seed=0, gamma=0.99, lr=5e-4, batch_size=64, buffer_size=20_000,
              learning_starts=1000, tau=0.005, eps_start=1.0, eps_end=0.05, eps_decay_steps=5_000):
    set_seed(seed)
    rng = np.random.default_rng(seed)
    env = CartPole()
    q_net = nn.Sequential(nn.Linear(4, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 2))
    target_net = nn.Sequential(nn.Linear(4, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 2))
    target_net.load_state_dict(q_net.state_dict())
    opt = torch.optim.Adam(q_net.parameters(), lr=lr)
    buf = ReplayBuffer(buffer_size, 4)
    obs, _ = env.reset(seed=seed)
    ep_ret, ep_returns, ep_end_steps = 0.0, [], []
    for step in range(total_steps):
        # ε 线性衰减：一开始纯随机探索，之后越来越依赖 Q 网络
        eps = max(eps_end, eps_start - (eps_start - eps_end) * step / eps_decay_steps)
        if rng.random() < eps:
            a = int(rng.integers(2))
        else:
            with torch.no_grad():
                a = int(q_net(torch.as_tensor(obs)).argmax())
        next_obs, r, term, trunc, _ = env.step(a)
        buf.add(obs, a, r, next_obs, float(term))       # 注意：存的是 terminated，不是 trunc
        obs, ep_ret = next_obs, ep_ret + r
        if term or trunc:
            ep_returns.append(ep_ret); ep_end_steps.append(step)
            obs, _ = env.reset(); ep_ret = 0.0
        if step >= learning_starts:
            loss = dqn_loss(q_net, target_net, buf.sample(batch_size, rng), gamma)
            opt.zero_grad(); loss.backward(); opt.step()
            # 软更新目标网络：w_T <- τ w + (1-τ) w_T（也可以每隔 C 步硬拷贝一次，原版 DQN 就是那样做的）
            with torch.no_grad():
                for p_t, p in zip(target_net.parameters(), q_net.parameters()):
                    p_t.mul_(1 - tau).add_(p, alpha=tau)
    return q_net, ep_returns, ep_end_steps

q_net, rets, ends = train_dqn()
plt.figure(figsize=(6, 3.5))
plt.plot(ends, rets, ".", alpha=0.3, label="episode return"); plt.plot(ends[19:], moving_average(rets, 20), label="moving avg (20)")
plt.xlabel("env step"); plt.ylabel("return (max 500)"); plt.legend(); plt.title("DQN on CartPole"); plt.show()
print("最后 20 局平均回报:", np.mean(rets[-20:]))

In [ ]:
# 测试贪心策略（不再探索）
def evaluate_q(q_net, n=10, seed=100):
    env, total = CartPole(), []
    for i in range(n):
        obs, _ = env.reset(seed=seed + i); done, R = False, 0.0
        while not done:
            with torch.no_grad():
                a = int(q_net(torch.as_tensor(obs)).argmax())
            obs, r, term, trunc, _ = env.step(a); R += r; done = term or trunc
        total.append(R)
    return np.mean(total)

score = evaluate_q(q_net)
print("贪心策略 10 局平均回报:", score)
assert score > 100, "DQN 没训练好：DQN 对随机种子很敏感，可以换个 seed 或者增加 total_steps 再试试"

> DQN 的训练曲线**很不稳定**，回报经常突然掉下去再回来。这是函数近似 + bootstrapping + off-policy 三者同时出现时的典型现象（Sutton 称之为"致命三角"）。后面的 Double DQN、SAC 的双 Q 网络、RLT 的 twin Q 都是在对付它。

---

## 小结

| 组件 | 作用 | 在 SAC / RLT 中 |
|---|---|---|
| Q 网络 | 用神经网络近似 $q(s,a)$ | critic（twin Q） |
| TD 目标 + `no_grad` | 半梯度：目标当常数 | 相同 |
| 目标网络 | 稳定 TD 目标 | 软更新 $\tau=0.005$ |
| 经验回放 | 打破相关性、复用数据 | replay buffer（RLT 还混入人类干预数据） |
| terminated vs truncated | 正确处理"有没有未来" | 相同 |

**DQN 的局限**：$\max_{a'}$ 只能在**离散**动作上算。机器人动作是连续的，怎么办？两条路：(1) 直接学一个策略网络输出动作（策略梯度，下一课）；(2) 学一个 actor 来近似 $\arg\max_a q$（DDPG/TD3/SAC，2.10 与 2.12 课）。